In [1]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F

# 1. Устройство `torch.Tensor`

## Трёхуровневая модель

Внутри PyTorch (на уровне C++) тензор - это цепочка:

```
Python-объект Tensor
      │
      ▼
TensorImpl  ──►  "Метаданные" (shape, stride, offset, dtype, device, autograd)
      │
      ▼
Storage (StorageImpl)  ──►  "Сырая память" (указатель на буфер, размер в байтах, аллокатор)
      │
      ▼
Физическая память (RAM / VRAM)
```

 - https://github.com/pytorch/pytorch/blob/main/c10/core/TensorImpl.h#L1717   (строка 569) 
 - https://github.com/pytorch/pytorch/blob/main/c10/core/StorageImpl.h

### Общая иерархия (сверху вниз)

```
Python-объект Tensor (torch.Tensor)
   │  ← тонкая обёртка, __dict__ почти пуст
   ▼
TensorImpl  ← «мозг»: метаданные и правила
   │
   ├──► Storage  ← handle (8 байт, ОДНО поле - intrusive_ptr)
   │       │
   │       ▼
   │    StorageImpl  ← «тело»: владеет буфером + метаданные памяти
   │       │
   │       ├──► DataPtr  ← unique-владелец сырого буфера
   │       │       │
   │       │       ▼
   │       │    [RAM / VRAM: N байт]
   │       │
   │       └──► Allocator*  ← «фабрика» памяти (не владеет)
   │
   ├──► AutogradMeta  ← опционально: grad_fn, grad, requires_grad
   ├──► NamedTensorMeta  ← опционально: имена измерений
   ├──► PyObjectSlot  ← связь с Python-объектом
   └──► DispatchKeySet  ← «маршрутизатор» операций (CPU/CUDA/Autograd/...)
```

### Уровень 1: `TensorImpl` — «мозг»

Содержит **всё, что описывает, как интерпретировать память**. Не хранит данные.

```
┌─────────────────────────────────────────────────────────────┐
│ TensorImpl : public intrusive_ptr_target                    │
│                                                             │
│  МЕТАДАННЫЕ ПРЕДСТАВЛЕНИЯ:                                  │
│   • sizes_and_strides_   ← shape + stride (в элементах)     │
│   • storage_offset_      ← с какого элемента Storage старт  │
│   • numel_               ← prod(sizes)                      │
│                                                             │
│  ТИП И УСТРОЙСТВО:                                          │
│   • data_type_ (TypeMeta) ← dtype (float32, int64, ...)     │
│   • device_opt_           ← cpu / cuda:0 / mps / meta       │
│   • key_set_ (DispatchKeySet) ← какие ядра применять        │
│                                                             │
│  ВЛАДЕНИЕ ПАМЯТЬЮ:                                          │
│   • storage_ (Storage)                                      │
│                                                             │
│  ОПЦИОНАЛЬНЫЕ УКАЗАТЕЛИ:                                    │
│   • autograd_meta_ (intrusive_ptr<AutogradMetaInterface>)   │
│   • named_tensor_meta_ (intrusive_ptr<...>)                 │
│   • pyobj_slot_ (impl::PyObjectSlot)                        │
│                                                             │
│  СЛУЖЕБНОЕ:                                                 │
│   • version_counter_ (VariableVersion) ← для in-place       │
│   • refcount_/weakcount_ (от intrusive_ptr_target)          │
└─────────────────────────────────────────────────────────────┘
```

**Что даёт:** `shape + stride + storage_offset + dtype` полностью описывают, **как тензор смотрит** на `Storage`. `DispatchKeySet` выбирает, **какое ядро** выполнять. `AutogradMeta` (если есть) — **как считать градиенты**.

### Уровень 2: `Storage` (handle)

Ровно **одно поле**, 8 байт.

```
┌──────────────────────────────────────────┐
│ Storage                                  │
│                                          │
│   • storage_impl_                        │
│     (intrusive_ptr<StorageImpl>)         │
│                                          │
│   Все методы — прокси:                   │
│     data()   → storage_impl_->data()     │
│     nbytes() → storage_impl_->nbytes()   │
│     device() → storage_impl_->device()   │
└──────────────────────────────────────────┘
```

**Что даёт:** дешёвый (8 байт), **копируемый** handle. При копировании `intrusive_ptr` делает `refcount++` на `StorageImpl`. Именно так много `TensorImpl` (view-ов) смотрят на один `StorageImpl`.

### Уровень 3: `StorageImpl` 

Создаётся **один раз** на буфер. Владеет памятью.

```
┌─────────────────────────────────────────────────────────┐
│ StorageImpl : public intrusive_ptr_target               │
│                                                         │
│  ВЛАДЕНИЕ ПАМЯТЬЮ:                                      │
│   • data_ptr_ (DataPtr)   ← UNIQUE владелец буфера      │
│   • size_bytes_ (SymInt)  ← размер буфера в БАЙТАХ      │
│   • allocator_ (Allocator*) ← не владеет, для resize    │
│   • resizable_ (bool)                                   │
│                                                         │
│  РЕФКАУНТИНГ:                                           │
│   • refcount_  (atomic) ← сколько Storage-handle'ов     │
│   • weakcount_ (atomic) ← для weak_intrusive_ptr        │
└─────────────────────────────────────────────────────────┘
```

**Что даёт:** **единственный** владелец буфера (`data_ptr_`), знает размер в байтах, делает `resize` (через `allocator_`). `refcount_` = сколько `Storage` на него смотрят.

### Уровень 4: `DataPtr` 

Уникальный владелец сырой памяти. Не копируется, только перемещается.

```
┌──────────────────────────────────────────────────┐
│ DataPtr (unique, move-only)                      │
│                                                  │
│   • data    (void*)        ← адрес в RAM/VRAM    │
│   • ctx     (void*)        ← контекст для deleter│
│   • deleter (DeleterFnPtr) ← как освободить      │
│   • device  (Device)       ← где лежит память    │
└──────────────────────────────────────────────────┘
```

**Что даёт:** инкапсулирует **владение** буфером + **способ освобождения** (deleter: `free`, `cudaFree`, кастомный). Когда `DataPtr` умирает → deleter освобождает буфер.

### Уровень 5: Память

```
[RAM / VRAM: N байт]
```

Физический блок, на который указывает `DataPtr::data`.

---

### Полная схема с примером view

```python
t = torch.arange(12).reshape(3, 4)   # создаём тензор
a = t.t()                             # view
b = t[0]                              # ещё один view
```

Состояние после:

```
t ──► [TensorImpl_t] ──► [Storage_t] ──┐
       shape=(3,4)        (8 байт)     │
       stride=(4,1)                    │
       offset=0                        │
                                       ├──► [StorageImpl]   ← ОДИН объект
a ──► [TensorImpl_a] ──► [Storage_a] ──┤       refcount_ = 3
       shape=(4,3)        (8 байт)     │       size_bytes_ = 48
       stride=(1,3)                    │       allocator_ = CPU
       offset=0                        │         │
                                       │         ▼
b ──► [TensorImpl_b] ──► [Storage_b] ──┘       [DataPtr]   ← unique
       shape=(4,)         (8 байт)               │
       stride=(1,)                               ▼
       offset=0                              [48 байт RAM]
                                             [0,1,2,...,11]
```

**Три `TensorImpl`**, **три `Storage`-handle'а**, **один `StorageImpl`**, **один `DataPtr`**, **один буфер**.

### Что происходит при каждой операции

| Операция | `TensorImpl` | `Storage` | `StorageImpl` | `DataPtr` | Буфер |
|---|---|---|---|---|---|
| `torch.randn(3,4)` | новый | новый | **новый** (`refcount=1`) | **новый** | **новый** |
| `t.t()` | новый | новый (копия) | тот же (`refcount++`) | тот же | тот же |
| `t[0]` | новый | новый (копия) | тот же (`refcount++`) | тот же | тот же |
| `t + 1` | новый | новый | **новый** (результат) | **новый** | **новый** |
| `t.contiguous()` | новый | новый | **новый** (если нужна копия) | **новый** | **новый** |
| `t.expand(...)` | новый (stride=0) | новый (копия) | тот же (`refcount++`) | тот же | тот же |
| `t.detach()` | новый | новый (копия) | тот же (`refcount++`) | тот же | тот же |

**Правило:** *view-операции (`t.t()`, `t[0]`, `expand`, `detach`) — новый `TensorImpl`, но тот же `StorageImpl`. Материализующие (`t + 1`, `contiguous`) — новый `StorageImpl` и новый буфер.*


In [2]:
t = torch.randn(2, 3, requires_grad=True)
print(type(t))                 
print(t.__dict__)              

<class 'torch.Tensor'>
{}


In [3]:
print("shape       :", t.shape)
print("stride      :", t.stride())          # шаг в элементах storage
print("storage_offset:", t.storage_offset())
print("dtype       :", t.dtype)
print("device      :", t.device)
print("layout      :", t.layout)
print("requires_grad:", t.requires_grad)
print("is_leaf     :", t.is_leaf)
print("data_ptr    :", t.data_ptr()) 

shape       : torch.Size([2, 3])
stride      : (3, 1)
storage_offset: 0
dtype       : torch.float32
device      : cpu
layout      : torch.strided
requires_grad: True
is_leaf     : True
data_ptr    : 3815810072768


## Разбор полей `torch.Tensor` (на примере `torch.randn(2, 3, requires_grad=True)`)

Ниже - что означает каждое поле из вывода и **почему** оно такое.

## 1. `shape : torch.Size([2, 3])`

**Что это:** логическая форма тензора - сколько элементов по каждому измерению.

**Почему так:** мы создали `torch.randn(2, 3)` — 2 строки, 3 столбца. `torch.Size` - подкласс `tuple`, поэтому работает как кортеж (`t.shape[0] == 2`, распаковка `r, c = t.shape`).

**Зачем нужно:** форма определяет число логических элементов: `2 * 3 = 6`. Но **форма ≠ размер памяти** (см. `stride`, `storage_offset`).

**Важно:** форма может не совпадать с физическим расположением в памяти (например, после `transpose` форма `(3, 2)`, а память та же).

## 2. `stride : (3, 1)`

**Что это:** шаг в **элементах Storage** (не в байтах!), который нужно сделать, чтобы перейти на 1 позицию вперёд по соответствующему измерению.

**Формула адресации:**

    physical_index = storage_offset
                   + i * stride[0]
                   + j * stride[1]

Для нашего тензора элемент `(i, j)` лежит по адресу `0 + i*3 + j*1`.

**Почему `(3, 1)`:** после `torch.randn(2, 3)` тензор **row-major contiguous** (C-порядок). Чтобы перейти на следующую строку — пропускаем 3 элемента, на следующий столбец — 1 элемент.

**Зачем нужно:** это ключ к пониманию **view-операций**.
- `t.t()` → stride становится `(1, 3)`, данные не двигаются.
- `t.expand(4, 2, 3)` → в «размноженной» размерности stride становится `0`.
- `t[::2]` → stride удваивается.

Если `stride` совпадает с «естественным» для формы (row-major) - тензор `is_contiguous() == True`, и его можно быстро копировать/переинтерпретировать без пересчёта индексов.


### Тензор = память + координаты + правило перевода

Главная путаница в `stride`/`offset` — это смешение **трёх разных сущностей**, которые никак не связаны:

1. **Память (`Storage`)** — одномерная лента байтов/чисел. У неё есть **позиции**: `Storage[0]`, `Storage[1]`, … Это физические места в RAM.
2. **Логические координаты** — то, что вы пишете в квадратных скобках: `t[i, j]`. Это просто числа, у них нет «места».
3. **Правило** — формула `pos(i, j) = offset + i·stride[0] + j·stride[1]`. Она говорит, какой координате какая позиция памяти соответствует.

```
логические координаты       правило (stride, offset)        память (Storage)
   (0,0) ──────┐                                        ┌── Storage[0]
   (0,1) ──────┤                                        ├── Storage[1]
   (1,0) ──────┼──→  pos = offset + i·s₀ + j·s₁  ──→    ├── Storage[2]
   (1,1) ──────┤                                        ├── Storage[3]
      …        ┘                                        └── Storage[4]
```

**Стрелка одна** - из координат в память. `stride` и `offset` — параметры этой стрелки. Данные не двигаются никогда — меняется только правило.

---


### Пример: `shape=(4, 2)`, `stride=(3, 2)`, `offset=0`

In [4]:
base = torch.arange(12)    
base

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])


```
позиция:   0    1    2    3    4    5    6    7    8    9   10   11
значение:  0    1    2    3    4    5    6    7    8    9   10   11
```

Это **просто лента** — 12 позиций подряд. Никакой «формы» у неё нет.

### Логические координаты

Форма `(4, 2)` — это **координаты** `(i, j)`, где `i ∈ {0,1,2,3}`, `j ∈ {0,1}`:

```
(0,0)  (0,1)
(1,0)  (1,1)
(2,0)  (2,1)
(3,0)  (3,1)
```



### Правило перевода

```
pos(i, j) = 0 + i·3 + j·2
```

| координата | подстановка | **позиция в памяти** |
|---|---|---|
| (0,0) | 0 + 0·3 + 0·2 | **0** |
| (0,1) | 0 + 0·3 + 1·2 | **2** |
| (1,0) | 0 + 1·3 + 0·2 | **3** |
| (1,1) | 0 + 1·3 + 1·2 | **5** |
| (2,0) | 0 + 2·3 + 0·2 | **6** |
| (2,1) | 0 + 2·3 + 1·2 | **8** |
| (3,0) | 0 + 3·3 + 0·2 | **9** |
| (3,1) | 0 + 3·3 + 1·2 | **11** |



```
ПАМЯТЬ (лента, Storage):
pos:     0    1    2    3    4    5    6    7    8    9   10   11
val:     0    1    2    3    4    5    6    7    8    9   10   11
         ↑         ↑    ↑         ↑    ↑         ↑    ↑         ↑
         │         │    │         │    │         │    │         │
ЛОГИЧЕСКИЕ КООРДИНАТЫ, которые попали в эти позиции:
      (0,0)   (0,1)   (1,0)     (1,1) (2,0)    (2,1) (3,0)     (3,1)

Дырки (позиции Storage, НЕ читаемые тензором):
             1              4              7             10
```


Читаем `t[2, 1]`:

```
t[2, 1]   →   pos = 0 + 2·3 + 1·2 = 8   →   Storage[8]   →   значение 8
 ↑                ↑                            ↑
 координата    правило                    физическое место
```


In [5]:
t = torch.as_strided(base, size=(4, 2), stride=(3, 2), storage_offset=0)

print(t)

tensor([[ 0,  2],
        [ 3,  5],
        [ 6,  8],
        [ 9, 11]])


In [6]:
print("shape :", t.shape)              
print("stride:", t.stride())            
print("offset:", t.storage_offset())    

shape : torch.Size([4, 2])
stride: (3, 2)
offset: 0


In [7]:
def pos(t, i, j):
    return t.storage_offset() + i * t.stride(0) + j * t.stride(1)

for i in range(4):
    for j in range(2):
        p = pos(t, i, j)
        assert t[i, j].item() == base[p].item()
        print(f"t[{i},{j}] → Storage[{p}] = {base[p].item()}")

t[0,0] → Storage[0] = 0
t[0,1] → Storage[2] = 2
t[1,0] → Storage[3] = 3
t[1,1] → Storage[5] = 5
t[2,0] → Storage[6] = 6
t[2,1] → Storage[8] = 8
t[3,0] → Storage[9] = 9
t[3,1] → Storage[11] = 11


In [8]:
def who(base, t, p):
    """Какие координаты читают Storage[p]?"""
    hits = []
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            if pos(t, i, j) == p:
                hits.append((i, j))
    return hits

for p in range(12):
    r = who(base, t, p)
    print(f"Storage[{p:2d}] ← {r if r else '- дырка'}")

Storage[ 0] ← [(0, 0)]
Storage[ 1] ← - дырка
Storage[ 2] ← [(0, 1)]
Storage[ 3] ← [(1, 0)]
Storage[ 4] ← - дырка
Storage[ 5] ← [(1, 1)]
Storage[ 6] ← [(2, 0)]
Storage[ 7] ← - дырка
Storage[ 8] ← [(2, 1)]
Storage[ 9] ← [(3, 0)]
Storage[10] ← - дырка
Storage[11] ← [(3, 1)]


### Почему `t[::2]` меняет `stride`, но не память

In [9]:
t3 = t[::2]

| | `t` | `t3 = t[::2]` |
|---|---|---|
| shape | `(4, 2)` | `(2, 2)` |
| stride | `(3, 2)` | `(6, 2)` |
| offset | `0` | `0` |
| **память** | тот же `Storage` | тот же `Storage` |

Правило стало `pos(i, j) = 0 + i·6 + j·2`, где теперь `i ∈ {0, 1}`:

```
t3[0,0] → pos(0,0) = 0 → Storage[0]   (это t[0,0])
t3[1,0] → pos(1,0) = 6 → Storage[6]   (это t[2,0] — пропустили t[1,0])
```

**Stride вырос с 3 до 6**, потому что между соседними строками нового тензора теперь лежат **две** исходные строки по 3 позиции → `2 · 3 = 6`. Память не двигалась.

### Пример для закрепления

`shape = (2,)`, `stride = (3,)`, `offset = 1`. Лента `Storage = [10, 11, 12, 13, 14, 15, 16]`.

Правило: `pos(i) = 1 + i·3`, `i ∈ {0, 1}`.


In [10]:
base = torch.tensor([10, 11, 12, 13, 14, 15, 16])
s = torch.as_strided(base, size=(2,), stride=(3,), storage_offset=1)

print(s)                
 

tensor([11, 14])


```
Storage:  [10] [11] [12] [13] [14] [15] [16]
позиция:    0    1    2    3    4    5    6
                  ↑              ↑
                i=0            i=1
               (offset)
```

Позиции `2` и `3` (значения `12`, `13`) — «дырки», их тензор не читает.


## 3. `storage_offset : 0`

**Что это:** с какого элемента `Storage` начинается **логический нулевой элемент** тензора.

**Почему `0`:** тензор создан с нуля (`torch.randn(...)`), он занимает начало своего блока памяти — смещения нет.

**Когда было бы не `0`:** после среза.

    

`z` - это **view** в `x`, данные не копировались, просто «окно» начинается с 5-го элемента `Storage`.

**Итог:** `storage_offset` + `stride` + `shape` полностью описывают, как тензор «нарезан» из `Storage`.

In [11]:
x = torch.arange(12).reshape(3, 4)
z = x[1:3, 1:3]
print(z.storage_offset())  # 5 = 1*4 + 1

5


### 4. `dtype : torch.float32`

**Что это:** тип элементов — как интерпретировать байты в памяти (`float32`, `int64`, `bfloat16`, `bool`, ...).

**Почему `float32`:** это дефолт для `torch.randn` и вообще для вещественных тензоров в PyTorch (если не менять `torch.set_default_dtype`).

**Зачем нужно:** dtype определяет **размер одного элемента в байтах** (`float32` = 4 байта) и набор доступных операций. `Storage` - это **байтовый** буфер, а dtype «накладывает сетку» на эти байты. Один и тот же `Storage` можно переинтерпретировать как `float32` или `int32` (через `view(dtype)`), и данные будут читаться по-разному.

**Связь с `stride`:** stride считается **в элементах**, а не в байтах, именно чтобы абстрагироваться от размера dtype. Физический байтовый адрес: `data_ptr() + (i*stride[0] + j*stride[1] + offset) * itemsize`.

## 5. `device : cpu`

**Что это:** где физически лежат данные и где выполняются операции.

**Почему `cpu`:** тензор создан без `.to("cuda")` - лежит в оперативной памяти (откуда тут видеокарты(((((( )

**Зачем нужно:**
- `device` определяет **аллокатор** (кто выделял память) и **набор доступных ядер** (CPU-ядро vs CUDA-ядро).
- Внутри `TensorImpl` это отражается в `DispatchKeySet` (например, `CPU` vs `CUDA`) — система диспетчеризации по нему выбирает реализацию операции.
- `data_ptr()` на CPU — обычный адрес в RAM; на CUDA — адрес в памяти GPU (обращаться к нему из Python напрямую нельзя).

## 6. `layout : torch.strided`

**Что это:** **способ физического хранения** данных - как интерпретировать `Storage`.

**Почему `strided`:** обычный «плотный» тензор с адресацией через `stride`/`offset`. Это дефолт для `torch.randn`.

**Какие бывают:**
- `torch.strided` - обычный dense-тензор (stride + offset).
- `torch.sparse_coo` / `torch.sparse_csr` / ... - разреженные форматы (хранят индексы и значения, а не плотный буфер).
- `torch._mkldnn` - специальный layout для oneDNN.

**Зачем нужно:** layout - отдельная ось от `dtype` и `device`. Он говорит, **как устроен Storage** (плотный буфер vs набор индексов/значений) и, соответственно, какие операции эффективны. `strided` = «данные лежат линейно, а вид задаётся strides».

**Связь с `stride`:** для не-`strided` layout'ов `stride()` обычно не определён или бессмысленен - там другая модель адресации.

## 7. `requires_grad : True`

**Что это:** флаг «участвует ли тензор в графе autograd» — то есть нужно ли для него считать градиенты.

**Почему `True`:** мы явно передали `requires_grad=True` при создании.

**Зачем нужно:** включает `AutogradMeta` у этого `TensorImpl`. Все операции над таким тензором будут записываться в граф (создавать `grad_fn` у результатов), а `.backward()` сможет пройти по нему и накопить `.grad`.

**Тонкость:** флаг живёт **на уровне `TensorImpl`, а не `Storage`**. Поэтому `x` и `x.t()` могут иметь разные `requires_grad` и разную историю автограда, хотя делят память.

**Экономия:** если `requires_grad=False` (дефолт), `AutogradMeta` не создаётся - экономия памяти и времени на построение графа.

# 2. Основные операции над тензорами


Когда мы вызываем простую математическую функцию вроде `torch.sin(x)` или `x + y`, PyTorch проделывает огромную архитектурную работу под капотом. На уровне C++ библиотеки **ATen** (A Tensor Library) все операции делятся на три категории:

1. **Поэлементные (Pointwise):** `sin`, `cos`, `exp`, `add`, `mul`. Применяются к каждому элементу независимо.
2. **Редукции (Reductions):** `sum`, `mean`, `max`. Сжимают одну или несколько размерностей тензора.
3. **Линейная алгебра (BLAS/LAPACK):** `matmul`, `inverse`, `svd`. Сложные матричные вычисления.


## 1. Слой Диспетчеризации (The Dispatcher)

PyTorch поддерживает десятки типов устройств (CPU, CUDA, XLA, MPS) и множество типов данных (Float, Double, Int8). Чтобы не писать код функции `sin` под каждую комбинацию вручную, используется **Диспетчер (Dispatcher)**. Это динамическая система, которая в рантайме решает, какую низкоуровневую C++ функцию вызвать.

### Алгоритм работы Диспетчера:
* **Анализ Dispatch Key:** Диспетчер смотрит на метаданные тензора (`Device`, `Layout`, наличие Autograd). Если тензор лежит на GPU, генерируется ключ `DispatchKey::CUDA`.
* **Поиск в таблице:** Диспетчер заглядывает в глобальную таблицу зарегистрированных функций и находит математический «kernel метод» (kernel) для этого ключа.
* **Вызов ядра:** Вызов перенаправляется на конкретную реализацию - например, функцию, оптимизированную под векторные инструкции процессора (AVX) или под CUDA-ядра.


## 2. Архитектурный паттерн TensorIterator

Для автоматизации поэлементных операций и редукций в PyTorch встроен один из самых сложных механизмов - **`at::TensorIterator`**. Это централизованный класс-помощник, который берет на себя всю рутину управления памятью.

### Что делает TensorIterator автоматически:
* **Broadcasting (Транслирование размеров):** Если вы складываете тензоры размеров `(3, 1)` и `(1, 4)`, именно `TensorIterator` рассчитывает, как виртуально расширить их до `(3, 4)`.
* **Type Promotion (Приведение типов):** Если сложить `Float` и `Double`, он автоматически приведет вычисления к более точному типу `Double`.
* **Оптимизация обхода памяти (Contiguity):** Он анализирует шаги (`strides`) тензоров и выстраивает циклы обхода так, чтобы данные читались последовательно, максимизируя попадание в кэш процессора.
* **Параллелизм:** На CPU он автоматически разбивает вычисления на потоки через OpenMP или TBB.


https://github.com/pytorch/pytorch/blob/main/aten/src/ATen/TensorIterator.h

## 3. Выполнение на CPU: Векторизация (AVX/SIMD)

Когда `TensorIterator` сформировал последовательные куски памяти для CPU, управление передается низкоуровневым ядрам. Чтобы вычисления на процессоре не превращались в медленный поцикловый обход, PyTorch использует **векторизацию (SIMD - Single Instruction, Multiple Data)**.

### Как это устроено:
* **Аппаратные инструкции:** Современные процессоры (Intel, AMD, ARM) умеют выполнять одну команду над целой пачкой чисел за один такт. PyTorch задействует технологии AVX2, AVX-512 или ARM Neon.
* **Класс Vectorized<T>:** Внутри библиотеки ATen за это отвечает шаблонный класс `at::vec::Vectorized<T>`. Например, для типа `float` и инструкций AVX2 этот класс за раз загружает из `StorageImpl` регистр размером 256 бит (это ровно 8 чисел `float`).
* **Векторный синус:** Вместо вызова стандартной функции `std::sin()` для каждого числа по очереди, PyTorch вызывает специальную математическую функцию, которая высчитывает синусы сразу для всей пачки (8 или 16 элементов) за раз.

https://github.com/pytorch/pytorch/blob/main/aten/src/ATen/native/cpu/UnaryOpsKernel.cpp
https://github.com/pytorch/pytorch/blob/main/aten/src/ATen/native/cpu/BinaryOpsKernel.cpp


## 4. Выполнение на GPU: CUDA-ядра (Kernels)

Если диспетчер определил, что тензор находится на видеокарте, то `TensorIterator` адаптирует логику под параллельную архитектуру GPU.

### Как это устроено:
* **Сетка потоков (Grid & Blocks):** Видеокарта сильна параллелизмом. PyTorch рассчитывает общее количество элементов в `StorageImpl` и нарезает вычисления на тысячи мелких потоков (`threads`), объединенных в блоки (`blocks`).
* **Параллельный обход:** Каждый поток получает свой уникальный индекс элемента в памяти. Вместо циклов, каждый поток просто берет «свое» число из видеопамяти и применяет к нему функцию.
* **Аппаратные блоки SFU:** Внутри графических чипов NVIDIA есть специальные транзисторные блоки - **Special Function Units (SFU)**. Они аппаратно оптимизированы для мгновенного подсчета трансцендентных и тригонометрических функций (`sin`, `cos`, `log`). CUDA-ядро PyTorch отправляет инструкцию напрямую в эти блоки.


In [12]:
x = torch.ones(3, 1)  # Столбец
y = torch.ones(1, 4)  # Строка
z = x + y             # Результат будет (3, 4)
z

tensor([[2., 2., 2., 2.],
        [2., 2., 2., 2.],
        [2., 2., 2., 2.]])

# 3. In-place операции, Out-параметры и управление мутабельностью памяти

В целях экономии памяти на GPU и CPU PyTorch предоставляет механизмы для изменения данных «на месте» без создания новых блоков в `StorageImpl`. Это реализуется через два механизма: **In-place операции** (с нижним подчеркиванием, например `x.sin_()`) и **Out-параметры** (`torch.sin(x, out=y)`).

---

## 1. Как TensorIterator обрабатывает In-place и Out-параметры

Когда мы вызываем обычную операцию `y = torch.sin(x)`, `TensorIterator` аллоцирует под `y` новый `StorageImpl`. Но если мы вызываем `x.sin_()` или `torch.sin(x, out=x)`, конфигурация итератора меняется:

```cpp
// Псевдокод настройки конфигурации внутри ATen для x.sin_()
auto iter = TensorIteratorConfig()
  .add_output(x) // Выход указывает на x
  .add_input(x)  // Вход тоже указывает на x
  .build();
```

### Перекрытие указателей (Aliasing)
В момент вызова `.build()` класс `TensorIteratorBase` проверяет, не перекрываются ли указатели входа и выхода в памяти (**Aliasing Detection**). 
* Если выходной тензор (`out`) совпадает со входным (как в `sin_`), `TensorIterator` оптимизирует обход памяти: в цикле чтения/записи `loop2d_t` массив `data[0]` (указатель на выход) и `data[1]` (указатель на вход) будут содержать **один и тот же адрес в `StorageImpl`**.
* Потоки (AVX или CUDA-threads) читают значение из ячейки памяти, сразу же вычисляют синус и перезаписывают значение по тому же самому адресу.

---

## 2. Побочные эффекты In-place изменений под капотом

Прямая модификация `StorageImpl` без выделения новой памяти — это дешево, но она создает две серьезные проблемы, которые PyTorch решает на уровне C++ метаданных в `TensorImpl`.

### А. Счетчик версий (Version Counter)
Внутри `c10::TensorImpl` (в файле `TensorImpl.cpp`) есть специальное скрытое поле — счетчик версий (`version_counter`). 
* При создании тензора его версия равна `0`.
* При каждой In-place операции (`sin_()`, `add_()`, прямое присвоение по индексам) этот счетчик инкрементируется: `version_++`.

**Зачем это нужно?** Это критически важно для **Autograd** (графа автоматического дифференцирования). Если тензор `x` участвовал в вычислении графа (например, `z = x * 2`), а затем вы вызвали `x.sin_()`, старые значения `x` стерлись из `StorageImpl`. Когда начнется обратный проход (`backward()`), Autograd проверит `version_counter` и, увидев, что данные изменились, выбросит знаменитую ошибку:  
`RuntimeError: a view of a leaf Variable that requires gradient is being used in an in-place operation.`

### Б. Проблема разделяемых представлений (View Aliasing)
Поскольку операции типа `y = x.view(...)` или `y = x[0]` создают новый `TensorImpl`, делящий общий `StorageImpl`, вызов `y.sin_()` автоматически изменит данные и в тензоре `x`. 
`TensorIterator` не следит за тем, сколько тензоров смотрят на этот кусок памяти — он просто выполняет запись по указателю. Ответственность за отслеживание таких связей лежит на системе `Version Counter`, которая прокидывает уведомления по цепочке связанных представлений (Views).

---

## 3. Сводная таблица типов операций с точки зрения памяти

| Тип вызова | Пример кода | Создается ли новый `TensorImpl`? | Создается ли новый `StorageImpl`? | Изменяет ли `Version Counter`? |
| :--- | :--- | :--- | :--- | :--- |
| **Out-of-place** | `y = torch.sin(x)` | **Да** | **Да** (новые данные) | Нет |
| **View (Представление)** | `y = x.view(-1)` | **Да** | **Нет** (общий с `x`) | Нет |
| **In-place** | `x.sin_()` | **Нет** | **Нет** (перезапись `x`) | **Да** (`version_++`) |
| **Out-параметр** | `torch.sin(x, out=y)` | **Нет** (переиспользует `y`) | **Нет** (пишет в память `y`) | **Да** (для `y`) |

---

### Анатомия счетчика версий (Version Counter)
Класс `c10::VariableVersion` (объявленный в `c10/core/TensorImpl.h`) — это не просто обычная переменная типа `int`. Это потокобезопасная структура данных, внутри которой инкапсулирован **атомарный счетчик** (`std::atomic<uint32_t>`), обернутый в умный указатель (`c10::intrusive_ptr`).

**Почему это сделано именно так?**
Когда мы создаем "представление" тензора через `.view()`, `.transpose()` или срезы (`x[0]`), PyTorch создает новый объект `TensorImpl` с новыми шагами (`strides`). Однако этот новый `TensorImpl` должен разделять историю изменений с оригинальным тензором. 

Поэтому при создании `view` PyTorch **копирует указатель** на один и тот же объект `VariableVersion`. В результате исходный тензор и все его `view`-копии начинают смотреть на **один и тот же счетчик в памяти**.

* **Срез инкрементирует общий счетчик:** Если вы делаете `view_tensor.add_(1)`, инкрементируется этот общий атомарный счетчик.
* **Autograd бьет тревогу:** Перед запуском вычисления градиентов для конкретной операции Autograd сохраняет у себя текущее числовое значение версии (например, `saved_version = 4`). На этапе `backward()` он проверяет: `if (current_version != saved_version)`. Если они не равны, это железное доказательство того, что данные внутри `StorageImpl` были перезаписаны *после* того, как граф зафиксировал операцию, и вычисления градиентов станут математически неверными.

### «Спецэффекты» разделяемого StorageImpl (Примеры из практики)

Поскольку `TensorIterator` слепо доверяет переданным ему шагам и указателям, работа с общим `StorageImpl` без выделения новой памяти может приводить к неожиданным побочным эффектам.


#### Пример 1: Изменение исходного тензора через срез («Классический побочный эффект»)
Когда мы создаем срез, физического копирования памяти не происходит. Новый тензор делит старый `StorageImpl`.

In [13]:
x = torch.zeros(3, 3)     # Создали матрицу 3х3 из нулей
row = x[0]                # Взяли первую строку (Shared Storage)

row.add_(5)               # In-place изменили строку
print(x)                  # Эффект: матрица x ТОЖЕ изменилась!


tensor([[5., 5., 5.],
        [0., 0., 0.],
        [0., 0., 0.]])


#### Пример 2: Наложение памяти при In-place бродкастинге (Курьезный случай)
Что произойдет, если мы попытаемся сделать in-place операцию, где один и тот же тензор бродкастится сам на себя?

In [14]:
x = torch.tensor([1.0, 2.0, 3.0])  # Вектор размера (3,)
# Пытаемся прибавить x (размер 3) in-place к матрице размера (3, 3)
# созданной через расширение осей (expand) того же самого x
matrix_view = x.expand(3, 3) 

matrix_view.add_(x) # ОШИБКА РАНТАЙМА!

RuntimeError: unsupported operation: more than one element of the written-to tensor refers to a single memory location. Please clone() the tensor before performing the operation.

*Что происходит под капотом:* 
1. Метод `.expand(3, 3)` выставил `strides` для первой оси в `0`. Физически в `StorageImpl` лежат всего 3 элемента, хотя логически мы видим матрицу 3х3.
2. При вызове `matrix_view.add_(x)` управление переходит в метод `TensorIteratorBase::compute_mem_overlaps()`.
3. `TensorIterator` анализирует шаги тензора, в который ведется запись (`matrix_view`). Он видит, что из-за нулевого шага **несколько логических элементов матрицы (`more than one element`) ссылаются на одну и ту же физическую ячейку памяти (`single memory location`)**.
4. Если бы PyTorch разрешил эту операцию, первый ряд потоков (или SIMD-инструкций) перезаписал бы память вектора `x`. Когда до вычислений дошли бы второй и третий ряд, они бы читали из `StorageImpl` уже испорченные, измененные данные, что привело бы к хаосу в математике.
5. Защитный механизм `TensorIterator` распознает это опасное перекрытие (aliasing) и аварийно завершает программу, требуя явно вызвать `.clone()`, чтобы разорвать связь и выделить под матрицу честный, независимый `StorageImpl`.


### Пример 3: Защитный механизм .clone()
Чтобы разорвать связь между тензорами и их общим `StorageImpl`, используется метод `.clone()`.

In [15]:

x = torch.ones(3, 3)
y = x.view(-1).clone() # Полное копирование StorageImpl

y.add_(1) # x остается нетронутым, так как у y теперь свой независимый StorageImpl

tensor([2., 2., 2., 2., 2., 2., 2., 2., 2.])

# 4. Устройство autograd

В лекции мы разобрали *математику* autograd: обратный ход — это цепочка VJP по графу, где каждый узел — объект `Function` с методом `backward(grad_output)`. Здесь спускаемся уровнем ниже: **как autograd физически устроен в C++-слое PyTorch** — где хранится граф, как он строится на forward-проходе и кто его исполняет при `backward()`.

```
TensorImpl (глава 1)
   └── autograd_meta_ → AutogradMeta  ← grad, grad_fn, requires_grad
          │
          ▼
Dispatcher (глава 2)
   ├── DispatchKey::Autograd         ← «регистратор»: строит граф
   ├── DispatchKey::ADInplaceOrView  ← сопровождение view и version counter
   └── DispatchKey::CPU / CUDA       ← настоящее вычисление
          │
          ▼
Node / Edge / SavedVariable         ← из чего физически состоит граф
          │
          ▼
torch::autograd::Engine             ← исполняет граф в обратном порядке
```

*План главы:* autograd как **ключ диспетчеризации** → объекты графа (**`Node`**, **`Edge`**, **`SavedVariable`**) → приёмник градиентов (**`AccumulateGrad`**) → исполняющий движок (**`Engine`**).


## 1. Autograd — это ещё один ключ диспетчеризации

В главе 2 мы видели, что каждая операция проходит через **Dispatcher**, который по `DispatchKeySet` выбирает ядро. Autograd встроен в тот же механизм: у *любого обычного* тензора в `key_set_` уже лежат ключи `AutogradCPU` и `ADInplaceOrView` — независимо от того, включён ли `requires_grad`.

```
Python:  y = x.sin()
   │
   ▼  Dispatcher смотрит на key_set_
DispatchKey::Autograd  →  «регистратор» (генерируемый VariableType-код)
   │   1) включён GradMode и хотя бы один вход требует градиента?
   │      да     → создать узел SinBackward0, соединить его Edge-ями
   │              с узлами входов, пришить к результату: y.grad_fn = узел
   │      нет    → просто пропустить вызов дальше (fall through)
   ▼
DispatchKey::ADInplaceOrView  →  «бухгалтерия»:
   │      для in-place: version_++, для view: учёт связей (см. главу 3)
   ▼
DispatchKey::CPU  →  at::native::sin()
       TensorIterator + Vectorized<float> (AVX) — собственно вычисление
```

То есть autograd **не перехватывает Python и не переписывает код** — это обычное ядро на обычном ключе диспетчеризации. Отличие лишь одно: оно *сначала* записывает операцию в граф, а *затем* передаёт вызов на backend.

### GradMode — решение «писать ли в граф»

Включена ли запись графа, определяется thread-local флагом **`GradMode`**:

- `no_grad()` — GradMode выключен: узлы не создаются вовсе, операция сразу уходит на CPU/CUDA-ядро;
- `enable_grad()` — принудительное включение (используют внутри библиотек, чтобы градиент считался и во «вложенных» вычислениях);
- `inference_mode()` — ещё радикальнее: тензор помечается ключом `InferenceMode`, autograd-ключи полностью удаляются из `key_set_` ⇒ минимальный overhead, но такие тензоры нельзя вернуть в autograd.

 - https://github.com/pytorch/pytorch/blob/main/torch/csrc/autograd/grad_mode.h


In [16]:
x = torch.randn(3, requires_grad=True)

# autograd-ключи лежат в маршруте ЛЮБОГО обычного тензора (autograd - часть dispatcher)
print("обычный        :", torch._C._dispatch_key_set(x))

with torch.no_grad():                     # GradMode выключен - узлы не создаются
    y = x.sin()
print("no_grad        : requires_grad =", y.requires_grad, "| is_leaf =", y.is_leaf,
      "| key_set =", torch._C._dispatch_key_set(y))

with torch.inference_mode():              # autograd исключён из маршрута совсем
    inf = x.sin()
print("inference_mode : requires_grad =", inf.requires_grad, "| key_set =", torch._C._dispatch_key_set(inf))

with torch.enable_grad():                 # GradMode включён - autograd пришивает узел
    z = x.sin()
print("enable_grad    : requires_grad =", z.requires_grad, "| is_leaf =", z.is_leaf,
      "| grad_fn =", type(z.grad_fn).__name__)

обычный        : DispatchKeySet(CPU, ADInplaceOrView, AutogradCPU, AutocastCPU)
no_grad        : requires_grad = False | is_leaf = True | key_set = DispatchKeySet(CPU, ADInplaceOrView, AutogradCPU, AutocastCPU)
inference_mode : requires_grad = False | key_set = DispatchKeySet(CPU, AutocastCPU)
enable_grad    : requires_grad = True | is_leaf = False | grad_fn = SinBackward0


**Что видно в выводе:**

- ключи `AutogradCPU` и `ADInplaceOrView` есть в маршруте самого обычного CPU-тензора: autograd **встроен в диспетчер**, а не «навешан» сверху;
- при `no_grad` ключи **не меняются** — отличие только в флаге `GradMode`, который autograd-ядро проверяет *в момент вызова*;
- при `inference_mode` autograd-ключи **исчезают** из `key_set_` — это единственный случай, когда маршрут реально «сокращается».

---

## 2. AutogradMeta — «блок autograd» у TensorImpl

В главе 1 в схеме `TensorImpl` было поле `autograd_meta_` — указатель на структуру **`AutogradMeta`**. Это **всё autograd-состояние тензора в одном объекте**:

```
┌────────────────────────────────────────────────────────────────┐
│ AutogradMeta   (torch/csrc/autograd/variable.h)                │
│                                                                │
│  grad_             (Variable)             ← то самое .grad     │
│  grad_fn_          (intrusive_ptr<Node>)  ← то самое .grad_fn  │
│  grad_accumulator_ (weak_ptr<Node>)   ← приёмник (см. разд. 4) │
│  requires_grad_    (bool)              ← флаг для листьев      │
│  output_nr_        (uint32) — «выход N» родителя породил тензор│
│  is_view_ / retains_grad_ / hooks  — вспомогательные           │
└────────────────────────────────────────────────────────────────┘
```

Отсюда несколько следствий:

- **Ленивость.** Пока тензор не попал в градиентную историю, `AutogradMeta` не создаётся вовсе (`autograd_meta_ == nullptr`) — та самая экономия из главы 1.
- `is_leaf` — не хранимый флаг, а **вычисляемое свойство**: лист — это просто тензор без `grad_fn`.
- `detach()` — новый `TensorImpl` с **тем же `Storage`**, но **без AutogradMeta**: история обрезана точно так же, как в главе 1 было про view-операции.

 - https://github.com/pytorch/pytorch/blob/main/torch/csrc/autograd/variable.h


In [25]:
x = torch.randn(2, requires_grad=True)
d  = x.sin()     # не-лист: узел пришит autograd-ядром
dd = d.detach()  # НОВЫЙ TensorImpl: тот же Storage, AutogradMeta отсутствует

print("d : is_leaf =", d.is_leaf,  "| requires_grad =", d.requires_grad,
      "| grad_fn =", type(d.grad_fn).__name__)
print("dd: is_leaf =", dd.is_leaf, "| requires_grad =", dd.requires_grad,
      "| grad_fn =", dd.grad_fn)

d : is_leaf = False | requires_grad = True | grad_fn = SinBackward0
dd: is_leaf = True | requires_grad = False | grad_fn = None


## 3. «Граф автограда» — это конкретные C++-классы: Node и Edge

Вершины графа — **`Node`**, рёбра — **`Edge`**:

```
┌─────────────────────────────────────────────────────────────────┐
│ Node  (torch/csrc/autograd/node.h)                              │
│                                                                 │
│  apply(inputs)    ← сам backward(grad_output) — VJP из лекции   │
│  next_edges_      ← список исходящих рёбер Edge                 │
│  input_metadata_  ← метаданные входов (dtype, устройство, shape)│
│  sequence_nr_     ← растёт в порядке построения графа;          │
│                     в очереди движка становится ПРИОРИТЕТОМ     │
│  name()           ← «SinBackward0», «AddBackward0», ...         │
└─────────────────────────────────────────────────────────────────┘

     Edge (torch/csrc/autograd/edge.h)
     ┌─────────────────────────────────────────────────┐
     │ function : Node*   ← узел-ПРИЁМНИК градиента    │
     │ input_nr : uint32  ← слот приёмника: номер его  │
     │                      входа, куда ляжет градиент │
     │                      (у листа всегда 0)         │
     └─────────────────────────────────────────────────┘
```

Правило fan-in прямо записано в комментарии `node.h`: *если из разных источников в один и тот же вход узла приходит несколько рёбер — значения суммируются до вызова узла*. Это не фигура речи, а конкретное устройство `InputBuffer` (раздел 6).

В Python `.next_functions` — это как раз список рёбер: пары `(Node | None, input_nr)`. `None` — константа (не тензор).

 - https://github.com/pytorch/pytorch/blob/main/torch/csrc/autograd/node.h
 - https://github.com/pytorch/pytorch/blob/main/torch/csrc/autograd/edge.h


In [26]:
a = torch.randn(2, requires_grad=True)
b = torch.randn(2, requires_grad=True)
st = torch.stack([a, b])        # StackBackward0: два входа — a (вход 0) и b (вход 1)
z = st.sin().sum()              # вниз от корня: sum -> sin -> stack

stack_node = z.grad_fn.next_functions[0][0].next_functions[0][0]
print("узел :", type(stack_node).__name__)
for fn, input_nr in stack_node.next_functions:
    print("Edge →", type(fn).__name__ if fn else None, "| input_nr =", input_nr)

w = 2 * a                       # лист a - операнд, 2 - константа
print(type(w.grad_fn).__name__, "→", [(type(fn).__name__ if fn else None, nr)
                                      for fn, nr in w.grad_fn.next_functions])

узел : StackBackward0
Edge → AccumulateGrad | input_nr = 0
Edge → AccumulateGrad | input_nr = 0
MulBackward0 → [('AccumulateGrad', 0), (None, 0)]


## 4. AccumulateGrad — куда физически пишется градиент

Оба ребра из примера выше ведут в узлы с одним и тем же именем — **`AccumulateGrad`**. У листа нет `grad_fn`, поэтому autograd-ядро записывает в его рёбра этот **приёмник**: узел ничего не вычисляет, а лишь дописывает пришедший градиент в `AutogradMeta.grad_` (то самое `.grad` в Python).

Отсюда и название: градиенты **аккумулируются** — повторный `backward()` прибавляет к `.grad`, а не заменяет значение. Поэтому в цикле обучения перед каждым шагом вызывают `optimizer.zero_grad()`.

Деталь «под капотом»: в очереди движка у `AccumulateGrad` жёстко выставлен максимальный приоритет (`sequence_nr = UINT64_MAX`) — градиенты листьев должны проходить первыми, пока не высвобождены узлы с их `SavedVariable`-ами.

А второе ребро `MulBackward0` — `None`: константа `2` не является тензором, и её Edge пуст.

---

## 5. SavedVariable — «шпаргалка» узла

Формулам обратного хода нужны промежуточные значения forward: для `sin` это вход `u` ($\bar u = \bar v\cos u$). Узел хранит такие значения в объекте **`SavedVariable`** — снимке тензора на момент построения графа:

```
┌──────────────────────────────────────────────────────────────┐
│ SavedVariable  (torch/csrc/autograd/saved_variable.h)        │
│                                                              │
│  data_          ← сохранённое значение тензора               │
│  saved_version_ ← его version_counter на тот момент (гл. 3)  │
│  is_leaf_ / output_nr_ / hooks (pack/unpack)                 │
└──────────────────────────────────────────────────────────────┘
```

Здесь и «стык» с главой 3: `SavedVariable` помнит **версию**. На backward движок распаковывает снимок и сверяет `saved_version_` с текущим счётчиком версии `TensorImpl`. Если между forward и backward данные перепишены in-place — версии расходятся, и движок отказывается считать.

 - https://github.com/pytorch/pytorch/blob/main/torch/csrc/autograd/saved_variable.h


In [27]:
x = torch.tensor([1.0, 2.0], requires_grad=True)
u = x.sin()      # u - не-лист; u.grad_fn = SinBackward0
v = u.cos()      # CosBackward0 сохранил вход u как SavedVariable (версия 0)
u.add_(1.0)      # in-place переписал u: version_ → 1, grad_fn перешит на AddBackward0

try:
    v.sum().backward()
except RuntimeError as e:
    print("RuntimeError:", str(e).split(" Hint:")[0])

RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.FloatTensor [2]], which is output 0 of AddBackward0, is at version 1; expected version 0 instead.


**Разбор сообщения:**

- `output 0 of AddBackward0` — u после `add_()` носит grad_fn самой in-place операции: **in-place тоже «строит» узел** (механизм `rebase_history`);
- `is at version 1; expected version 0` — если буквально: движок сверил `saved_version_` снимка (0) с текущей версией `TensorImpl` (1, после `version_++`) — это ровно тот счётчик версий из главы 3.

---

## 6. Обратный ход изнутри: Engine, GraphTask, ReadyQueue, InputBuffer

Всё, что выше, происходило **на forward** — там граф только строился. Исполняет его **`torch::autograd::Engine`**:

```
loss.backward()
     │
     ▼
Engine::execute(root = loss.grad_fn, ...)
     │  1) создаётся GraphTask: keep_graph ← retain_graph,
     │     для узлов заранее считаются счётчики зависимостей
     │  2) стартовый узел оборачивается в GraphRoot, который принимает
     │     seed (для loss это градиент [1.]), и кладётся в ReadyQueue
     ▼
 worker thread (по одному на девайс, у каждого своя ReadyQueue)
     │  цикл:
     │   pop NodeTask  ← (узел fn_ + InputBuffer inputs_)
     │    • InputBuffer - implicit «add»: градиенты от разных источников,
     │      пришедшие в один слот, суммируются ещё до вызова узла (fan-in)
     │    • fn_(inputs_)              ← тот самый backward(grad_output)
     │    • раздача результатов по next_edges: edge.input_nr выбирает
     │      слот приёмника
     │    • dependencies[приёмник]--; стало 0 → узел готов → в очередь
     ▼
 AccumulateGrad → дописывает .grad листу
```

Ключевые детали:

- **Топология подсчитывается лениво.** Явной топологической сортировки нет: узел исполняется, когда его счётчик зависимостей дошёл до нуля, т.е. когда все «чужие» градиенты приехали. Ветки, приходящие в один канал, просто суммируются в `InputBuffer`.
- **Приоритет — это `sequence_nr`.** `ReadyQueue` — куча: узлы, построенные *позже* (больший `sequence_nr`), всплывают первыми, поэтому граф обходится почти в обратном порядке построения.
- **Потоки — по девайсам.** У каждого девайса (CPU, `cuda:0`, ...) свой worker и своя `ReadyQueue`; рёбра в другой девайс переносятся в чужую очередь. Отсюда многопоточный backward в multi-GPU.
- **Граф одноразовый.** По умолчанию `keep_graph = False`: после исполнения узел высвобождает свои `SavedVariable` (`release_variables`) — память свободна, но второй `backward()` уже невозможен.

 - https://github.com/pytorch/pytorch/blob/main/torch/csrc/autograd/engine.h
 - https://github.com/pytorch/pytorch/blob/main/torch/csrc/autograd/engine.cpp


In [28]:
xr = torch.randn(2, requires_grad=True)
z = (2 * xr + 3).sin().sum()
z.backward()                      # первый проход: keep_graph = False, граф разобран

try:
    z.backward()                  # второй проход: SavedVariable-ы уже высвобождены
except RuntimeError as e:
    print("RuntimeError:", str(e).split(". Saved")[0], "…")

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed) …


In [29]:
xf = torch.ones(1, requires_grad=True)
y = xf.sin() + xf.sin()    # ДВА разных узла SinBackward0, оба смотрят на один лист

y.backward()               # оба ребра приехали в ОДИН слот AccumulateGrad → implicit add
print(f"x.grad = {xf.grad.item():.7f} ≈ 2·cos(1) = {2*np.cos(1):.7f}")

x.grad = 1.0806047 ≈ 2·cos(1) = 1.0806046


---

## 7. Сводка: где что живёт

| В Python | Под капотом (C++) | Где в исходниках |
| :--- | :--- | :--- |
| `.grad_fn`, `.next_functions` | `intrusive_ptr<Node> grad_fn_`, `edge_list next_edges_` | `node.h`, `edge.h` |
| `.grad` | поле `grad_` внутри `AutogradMeta` | `variable.h` |
| `.is_leaf` | не хранится: вычисляется как «`grad_fn == nullptr`» | `variable.h` |
| построение графа | autograd-ядро на ключе `Autograd` (генерируемый `VariableType`-код) | `torch/csrc/autograd/generated/` |
| `backward()` | `Engine::execute` → `GraphTask` → `NodeTask` в `ReadyQueue` | `engine.h` |
| `retain_graph=True` | `GraphTask.keep_graph` — не высвобождать `SavedVariable` | `graph_task.h` |
| in-place ошибка на backward | рассинхрон `saved_version_` с `version_counter_` | `saved_variable.h`, глава 3 |
| `AccumulateGrad` | узел-приёмник, пишущий в `.grad` листа; приоритет `UINT64_MAX` | `functions/accumulate_grad.h` |
| `no_grad()` / `inference_mode()` | флаг `GradMode` / удаление autograd-ключей из `key_set_` | `grad_mode.h` |
| сумма градиентов из веток | `InputBuffer` — implicit add при fan-in | `input_buffer.h` |

**Итог главы.** Autograd — это не отдельный «модуль», а конструкция из трёх частей, сшитая диспетчером из глав 1-3:

1. **ядро на ключе `Autograd`** при каждой операции решает, строить ли граф, и строит его из `Node`/`Edge`;
2. **данные истории на тензорах** — `AutogradMeta` внутри `TensorImpl` (grad, grad_fn, счётчик версий) и `SavedVariable` в узлах;
3. **движок `Engine`** — многопоточный исполнитель, который обходит граф «ленивой топологией» через `ReadyQueue` и `InputBuffer`, а математику VJP из лекции подставляют методы `apply()` каждого узла.
